# Field Transformations Quickstart

`FieldTransformation(source, expr)` is the FeynPy analogue of a FeynRules field `Definitions` entry. Transformations act on **compiled interaction terms**, so covariant derivatives and field strengths are expanded in the original gauge basis first.

The Standard Model workflow is:

1. declare and compile the gauge-basis Lagrangian;
2. expand finite weak indices into explicit components;
3. apply one simultaneous field-transformation stage;
4. simplify parameter identities and extract physical-basis Feynman rules.

This notebook covers the current transformation surface:

| Feature | Form |
|---|---|
| Linear or charged mixing | `FieldTransformation(B, -sw * Z + cw * A)` |
| Component-selective rule | `components={index_slot: component}` |
| Vacuum shift | `vev / sqrt(2) + H / sqrt(2) + I * G0 / sqrt(2)` |
| Chiral/flavor matrices | `rotation(CKM, CKMDag) * ProjM * dq` |
| Hermitian conjugation | automatic for `source.bar`; use `real_symbols=` for real coefficients |
| Explicit replacement monomials | `terms=(replacement(coefficient, *fields), ...)` |
| Custom index-dependent rule | `builder(context)` and `context.fresh(...)` |
| Dependent definitions | `repeat=True` reaches a fixed point; cycles are rejected |

In [1]:
from __future__ import annotations

import re
import sys
from fractions import Fraction
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Could not locate the FeynPy repository root")
for entry in (ROOT, ROOT / "src"):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

from symbolica import Expression, S

from feynpy import (
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_FUND_INDEX,
    CompiledLagrangian,
    DC,
    Field,
    FieldTransformation,
    GaugeGroup,
    Model,
    Parameter,
    PartialD,
    ProjM,
    find_source_basis_occurrences,
    flavor_index,
    replacement,
    rotation,
    validate_compiled_index_multiplicities,
)
from feynpy.interactions import InteractionTerm
from models.SM import build_standard_model
from models.SM.SM_support import standard_model_weak_tensor_components

print("Repository root:", ROOT)

Repository root: /Users/rems/Documents/MScThesis


In [2]:
ANSI = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
DISPLAY_PREFIX = re.compile(r"(?:python|spenso(?:_python)?)::(?:\{[^}]*\}::)?")
DISPLAY_TAG = re.compile(r"\{[^}]*\}::")


def clean(value):
    """Render Symbolica and declared-DSL objects without backend namespaces."""
    if hasattr(value, "to_symbolica") and not hasattr(value, "to_canonical_string"):
        value = value.to_symbolica()
    rendered = (
        value.to_canonical_string()
        if hasattr(value, "to_canonical_string")
        else str(value)
    )
    rendered = ANSI.sub("", rendered)
    rendered = DISPLAY_PREFIX.sub("", rendered)
    return DISPLAY_TAG.sub("", rendered)


def occurrence_text(occurrence):
    suffix = ".bar" if occurrence.conjugated and not occurrence.field.self_conjugate else ""
    labels = [clean(label) for label in occurrence.slot_labels.values if label is not None]
    return f"{occurrence.field.name}{suffix}" + (f"[{', '.join(labels)}]" if labels else "")


def show_terms(lagrangian, *, limit=12):
    """Print the structural field content relevant to transformation examples."""
    print(f"terms: {len(lagrangian.terms)}")
    for number, term in enumerate(lagrangian.terms[:limit], start=1):
        fields = " · ".join(occurrence_text(item) for item in term.fields) or "1"
        derivatives = ""
        if term.derivatives:
            actions = ", ".join(
                f"d({clean(action.lorentz_index)})@{action.target}"
                for action in term.derivatives
            )
            derivatives = f"; {actions}"
        print(f"  {number:>2}: {clean(term.coupling)} × {fields}{derivatives}")
    if len(lagrangian.terms) > limit:
        print(f"  ... {len(lagrangian.terms) - limit} more")


def safe_rule(lagrangian, *fields):
    try:
        return lagrangian.feynman_rule(*fields)
    except ValueError:
        return Expression.num(0)

## 1. Compile first, then apply a linear mixing

Here a scalar kinetic term is compiled with the hypercharge gauge field `B`. The transformation is applied afterward, when `DC` has already generated the derivative, cubic, and quartic interaction structures.

In [3]:
mu = S("mu")
g1 = S("g1")
sw = S("sw")
cw = S("cw")
yH = S("yH")

B = Field("B", spin=1, self_conjugate=True, symbol=S("B0"), indices=(LORENTZ_INDEX,))
A = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Z = Field("Z", spin=1, self_conjugate=True, symbol=S("Z0"), indices=(LORENTZ_INDEX,))
Hdoublet = Field(
    "Hdoublet",
    spin=0,
    self_conjugate=False,
    symbol=S("Hdoublet0"),
    conjugate_symbol=S("Hdoubletdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": yH},
)

U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson="B",
    charge="Y",
)
source_model = Model(
    name="transform-quickstart",
    gauge_groups=(U1Y,),
    fields=(Hdoublet, B, A, Z),
    lagrangian_decl=DC(Hdoublet.bar, mu) * DC(Hdoublet, mu),
)
L_source = source_model.lagrangian()

print("Source-basis vertices")
for vector in (B, A, Z):
    print(f"  Γ(H†, H, {vector.name}) = {clean(safe_rule(L_source, Hdoublet.bar, Hdoublet, vector))}")

Source-basis vertices
  Γ(H†, H, B) = -1𝑖*g1*pcomp(q1,mu3)*yH*g(cof(2,w1),cof(2,w2))+1𝑖*g1*pcomp(q2,mu3)*yH*g(cof(2,w1),cof(2,w2))
  Γ(H†, H, A) = 0
  Γ(H†, H, Z) = 0


In [4]:
neutral_mixing = FieldTransformation(B, -sw * Z + cw * A)
L_physical = source_model.transform_fields(neutral_mixing, repeat=False)

print("Physical-basis vertices")
for vector in (B, A, Z):
    print(f"  Γ(H†, H, {vector.name}) = {clean(safe_rule(L_physical, Hdoublet.bar, Hdoublet, vector))}")

assert clean(safe_rule(L_physical, Hdoublet.bar, Hdoublet, B)) == "0"
assert clean(safe_rule(L_physical, Hdoublet.bar, Hdoublet, A)) != "0"
assert clean(safe_rule(L_physical, Hdoublet.bar, Hdoublet, Z)) != "0"

Physical-basis vertices
  Γ(H†, H, B) = 0
  Γ(H†, H, A) = -1𝑖*cw*g1*pcomp(q1,mu3)*yH*g(cof(2,w1),cof(2,w2))+1𝑖*cw*g1*pcomp(q2,mu3)*yH*g(cof(2,w1),cof(2,w2))
  Γ(H†, H, Z) = -1𝑖*g1*pcomp(q2,mu3)*sw*yH*g(cof(2,w1),cof(2,w2))+1𝑖*g1*pcomp(q1,mu3)*sw*yH*g(cof(2,w1),cof(2,w2))


## 2. Expand finite indices before component rules

A rule such as `components={0: 2}` matches a numeric value in index slot 0. A contracted symbolic weak index must therefore be unfolded with `expand_index_components(...)` first.

The two rules below mirror the SM Higgs-doublet definitions. The upper component becomes a charged Goldstone. The lower component contains a constant vacuum piece, the Higgs, and the neutral Goldstone. The same rules automatically transform `Phi.bar`, conjugating coefficients and fields.

In [5]:
HALF = Expression.num(1) / Expression.num(2)
INV_SQRT2 = HALF**HALF
vev = S("vev")

Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("Phi0"),
    conjugate_symbol=S("Phidag0"),
    indices=(WEAK_FUND_INDEX,),
)
GP = Field(
    "GP",
    spin=0,
    self_conjugate=False,
    symbol=S("GP0"),
    conjugate_symbol=S("GM0"),
)
Higgs = Field("Higgs", spin=0, self_conjugate=True, symbol=S("Higgs0"))
G0 = Field("G0", spin=0, self_conjugate=True, symbol=S("G00"))

scalar_source = Model(
    name="component-and-vacuum-shift",
    fields=(Phi, GP, Higgs, G0),
    lagrangian_decl=Phi.bar * Phi,
).lagrangian()
weak_expanded = scalar_source.expand_index_components(WEAK_FUND_INDEX)

print("Before component expansion")
show_terms(scalar_source)
print("\nAfter component expansion")
show_terms(weak_expanded)
assert len(weak_expanded.terms) == 2

Before component expansion
terms: 1
   1: 1 × Phi.bar[w_decl_1] · Phi[w_decl_1]

After component expansion
terms: 2
   1: 1 × Phi.bar[1] · Phi[1]
   2: 1 × Phi.bar[2] · Phi[2]


In [6]:
breaking_rules = (
    FieldTransformation(Phi, -Expression.I * GP, components={0: 1}),
    FieldTransformation(
        Phi,
        vev * INV_SQRT2 + INV_SQRT2 * Higgs + Expression.I * INV_SQRT2 * G0,
        components={0: 2},
    ),
)
broken_scalar = weak_expanded.transform_fields(
    *breaking_rules,
    repeat=False,
    real_symbols=(vev,),
)

show_terms(broken_scalar)
print("\nSelected rules")
for fields in ((GP.bar, GP), (Higgs,), (Higgs, Higgs), (G0, G0)):
    names = ", ".join(
        field.name if hasattr(field, "name") else field.field.name + ".bar"
        for field in fields
    )
    print(f"  Γ({names}) = {clean(safe_rule(broken_scalar, *fields))}")

assert all(occurrence.field is not Phi for term in broken_scalar.terms for occurrence in term.fields)
assert "conj(vev)" not in " ".join(clean(term.coupling) for term in broken_scalar.terms)

terms: 7
   1: 1 × GP.bar · GP
   2: 1/2*vev^2 × 1
   3: vev × Higgs
   4: 1/2 × Higgs · Higgs
   5: 1𝑖/2 × Higgs · G0
   6: -1𝑖/2 × G0 · Higgs
   7: 1/2 × G0 · G0

Selected rules
  Γ(GP.bar, GP) = 1𝑖
  Γ(Higgs) = 1𝑖*vev
  Γ(Higgs, Higgs) = 1𝑖
  Γ(G0, G0) = 1𝑖


## 3. Chiral projectors and flavor rotations

Matrix factors are wired to the source field's matching free index type. `ProjM` acts on the spinor index; `rotation(V, Vdag)` acts on the flavor index. Spectator indices, here color, are inherited.

The isolated compiled terms below make that wiring visible. Normal models produce these terms during compilation; direct `InteractionTerm` construction is only used here to avoid hiding the result inside a larger fermion Lagrangian.

In [7]:
generation = flavor_index("GenerationDemo", 3, prefix="gd")
QL = Field(
    "QL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("QL0"),
    conjugate_symbol=S("QLbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX, generation, COLOR_FUND_INDEX),
)
dq = Field(
    "dq",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("dq0"),
    conjugate_symbol=S("dqbar0"),
    indices=(SPINOR_INDEX, generation, COLOR_FUND_INDEX),
)
V = Parameter(
    "Vdemo",
    indices=(generation, generation),
    complex_param=True,
    unitary_partner="Vdagdemo",
)
Vdag = Parameter(
    "Vdagdemo",
    indices=(generation, generation),
    complex_param=True,
    unitary_partner="Vdemo",
)

matrix_source = CompiledLagrangian(
    terms=(
        InteractionTerm(coupling=1, fields=(QL(S("s"), 2, S("f"), S("c")),)),
        InteractionTerm(coupling=1, fields=(QL.bar(S("sb"), 2, S("fb"), S("cb")),)),
    ),
    parameters=(V, Vdag),
)
matrix_rule = FieldTransformation(
    QL,
    rotation(V, Vdag) * ProjM * dq,
    components={1: 2},
)
matrix_result = matrix_source.transform_fields(matrix_rule, repeat=False)
show_terms(matrix_result)

matrix_couplings = [clean(term.coupling) for term in matrix_result.terms]
assert any("PL(" in coupling and "Vdemo(" in coupling for coupling in matrix_couplings)
assert any("PR(" in coupling and "Vdagdemo(" in coupling for coupling in matrix_couplings)

terms: 2
   1: PL(s,i_canon_1)*Vdemo(f,gd_canon_1) × dq[i_canon_1, gd_canon_1, c]
   2: PR(i_canon_1,sb)*Vdagdemo(gd_canon_1,fb) × dq.bar[i_canon_1, gd_canon_1, cb]


For the barred occurrence the engine reverses the matrix chain, changes `ProjM` to the conjugate projector, uses `Vdag`, and produces `dq.bar`. Coefficients listed in `real_symbols=` are left real during this automatic conjugation.

Expression-style matrix rules record their target fields as dependencies, so fixed-point ordering and cycle detection also work for projector/rotation definitions.

## 4. Explicit replacement terms and custom builders

The expression DSL is the default. Two lower-level forms cover cases it cannot express:

- `terms=` accepts explicit `replacement(coefficient, *fields)` monomials, including a product of replacement fields. Derivatives obey the Leibniz rule over such a product.
- `builder=` receives an occurrence-specific `TransformationContext`. It can inspect source labels and allocate collision-free dummy indices with `context.fresh(...)`. A builder should declare `dependencies=` for generated fields and `conjugate_builder=` when it must handle barred sources.

In [8]:
phi = Field("phi", spin=0, self_conjugate=True, symbol=S("phi0"))
x = Field("x", spin=0, self_conjugate=True, symbol=S("x0"))
y = Field("y", spin=0, self_conjugate=True, symbol=S("y0"))

derivative_source = Model(
    name="product-replacement",
    fields=(phi, x, y),
    lagrangian_decl=PartialD(phi, mu),
).lagrangian()
leibniz_result = derivative_source.transform_fields(
    FieldTransformation(phi, terms=(replacement(1, x, y),)),
    repeat=False,
)
show_terms(leibniz_result)

assert len(leibniz_result.terms) == 2
assert {term.derivatives[0].target for term in leibniz_result.terms} == {0, 1}

terms: 2
   1: 1 × x · y; d(mu)@0
   2: 1 × x · y; d(mu)@1


In [9]:
seed = Field("seed", spin=0, self_conjugate=True, symbol=S("seed0"))
colored_x = Field(
    "colored_x", spin=0, self_conjugate=True, symbol=S("colored_x0"), indices=(COLOR_FUND_INDEX,)
)
colored_y = Field(
    "colored_y", spin=0, self_conjugate=True, symbol=S("colored_y0"), indices=(COLOR_FUND_INDEX,)
)


def color_pair_builder(context):
    color = context.fresh(COLOR_FUND_INDEX, "pair")
    return (replacement(S("kappa"), colored_x(color), colored_y(color)),)


builder_source = Model(
    name="builder-example",
    fields=(seed, colored_x, colored_y),
    lagrangian_decl=seed,
).lagrangian()
builder_result = builder_source.transform_fields(
    FieldTransformation(
        seed,
        builder=color_pair_builder,
        dependencies=(colored_x, colored_y),
        auto_conjugate=False,
    ),
    repeat=False,
)
show_terms(builder_result)

left_label, right_label = (item.slot_labels.get(0) for item in builder_result.terms[0].fields)
assert left_label == right_label

terms: 1
   1: kappa × colored_x[c_canon_1] · colored_y[c_canon_1]


## 5. Simultaneous passes and dependent definitions

Within one pass, replacements created by one rule are not consumed by another rule. `repeat=False` therefore performs exactly one simultaneous definitions stage—the mode used by the SM. `repeat=True` repeats stages until no source fields remain. Dependency cycles raise `CyclicTransformationError`, and `max_passes` guards fixed-point evaluation.

In [10]:
stage_a = Field("stage_a", spin=0, self_conjugate=True, symbol=S("stage_a0"))
stage_b = Field("stage_b", spin=0, self_conjugate=True, symbol=S("stage_b0"))
stage_c = Field("stage_c", spin=0, self_conjugate=True, symbol=S("stage_c0"))
stage_source = Model(
    name="dependent-transformations",
    fields=(stage_a, stage_b, stage_c),
    lagrangian_decl=stage_a,
).lagrangian()
stage_rules = (
    FieldTransformation(stage_a, stage_b),
    FieldTransformation(stage_b, stage_c),
)

one_pass = stage_source.transform_fields(*stage_rules, repeat=False)
fixed_point = stage_source.transform_fields(*stage_rules, repeat=True)
print("repeat=False:", occurrence_text(one_pass.terms[0].fields[0]))
print("repeat=True :", occurrence_text(fixed_point.terms[0].fields[0]))

assert one_pass.terms[0].fields[0].field is stage_b
assert fixed_point.terms[0].fields[0].field is stage_c

repeat=False: stage_b
repeat=True : stage_c


## 6. The complete Standard Model usage

`models/SM/SM_support.py::compile_source_piece` applies the production pipeline to each sector:

```python
component_lagrangian = source_piece.lagrangian().expand_index_components(
    WEAK_FUND_INDEX,
    WEAK_ADJ_INDEX,
    tensor_components=standard_model_weak_tensor_components(),
)
broken_piece = component_lagrangian.transform_fields(
    *transformations,
    repeat=False,
    real_symbols=real_symbols,
)
```

The extra component map supplies explicit Pauli-generator, SU(2) structure-constant, and antisymmetric-tensor values that cannot be inferred from index ranges alone.

In [11]:
weak_tensor_components = standard_model_weak_tensor_components()
print("Explicit weak tensor components:", len(weak_tensor_components))
for pattern, value in list(weak_tensor_components.items())[:6]:
    print(f"  {clean(pattern)} -> {clean(value)}")

Explicit weak tensor components: 43
  t(coad(3,1),cof(2,1),cof(2,1)) -> 0
  t(coad(3,1),cof(2,1),cof(2,2)) -> 1/2
  t(coad(3,1),cof(2,2),cof(2,1)) -> 1/2
  t(coad(3,1),cof(2,2),cof(2,2)) -> 0
  t(coad(3,2),cof(2,1),cof(2,1)) -> 0
  t(coad(3,2),cof(2,1),cof(2,2)) -> -1𝑖/2


In [12]:
sm = build_standard_model()

print("Gauge basis -> physical basis definitions")
header = f"{'source':<8} {'fixed components':<20} replacement"
print(header)
print("-" * len(header))
for transformation in sm.transformations:
    components = str(dict(transformation.components)) if transformation.components else "all"
    print(
        f"{transformation.source.name:<8} "
        f"{components:<20} "
        f"{clean(transformation.expr)}"
    )
print(f"\nTotal definitions: {len(sm.transformations)}")

Gauge basis -> physical basis definitions
source   fixed components     replacement
-----------------------------------------
B        all                  -sw * Z + cw * A
Wi       {1: 1}               (1/2)^(1/2) * W.bar + (1/2)^(1/2) * W
Wi       {1: 2}               -1𝑖*(1/2)^(1/2) * W.bar + 1𝑖*(1/2)^(1/2) * W
Wi       {1: 3}               cw * Z + sw * A
Phi      {0: 1}               -1𝑖 * GP
Phi      {0: 2}               (1/2)^(1/2)*vev + (1/2)^(1/2) * H + 1𝑖*(1/2)^(1/2) * G0
LL       {1: 1}               ProjM * vl
LL       {1: 2}               ProjM * l
lR       all                  ProjP * l
QL       {1: 1}               ProjM * uq
QL       {1: 2}               CKM * ProjM * dq
uR       all                  ProjP * uq
dR       all                  ProjP * dq
ghB      all                  -sw * ghZ + cw * ghA
ghWi     {0: 1}               (1/2)^(1/2) * ghWp + (1/2)^(1/2) * ghWm
ghWi     {0: 2}               1𝑖*(1/2)^(1/2) * ghWp + -1𝑖*(1/2)^(1/2) * ghWm
ghWi     {0: 3}         

In [13]:
transformed_source_fields = (
    sm.fields.LL,
    sm.fields.lR,
    sm.fields.QL,
    sm.fields.uR,
    sm.fields.dR,
    sm.fields.Phi,
    sm.fields.B,
    sm.fields.Wi,
    sm.fields.ghB,
    sm.fields.ghWi,
)
residual_source_fields = find_source_basis_occurrences(
    sm.lagrangian,
    source_fields=transformed_source_fields,
)
index_issues = validate_compiled_index_multiplicities(sm.lagrangian)

print("Compiled physical terms       :", len(sm.lagrangian.terms))
print("Unexpanded vertex signatures :", len(sm.lagrangian.vertex_signatures()))
print("Flavor-expanded signatures   :", len(sm.lagrangian.vertex_signatures(flavor_expand=True)))
print("Residual transformed sources  :", len(residual_source_fields))
print("Index-multiplicity issues     :", len(index_issues))

assert residual_source_fields == ()
assert index_issues == ()

Compiled physical terms       : 306
Unexpanded vertex signatures : 123


Flavor-expanded signatures   : 201
Residual transformed sources  : 0
Index-multiplicity issues     : 0


## 7. Practical checklist

- Compile gauge-covariant declarations before transforming fields. `Model.transform_fields(...)` does this automatically.
- Expand every finite index used by a `components=` selector. Supply `tensor_components=` when concrete invariant tensors occur in couplings.
- Prefer the expression DSL for mixing, vacuum shifts, projectors, and rotations. Use `terms=` or `builder=` only for structures the DSL cannot represent.
- Pass known-real coefficients through `real_symbols=` so automatic barred-field rules do not introduce unnecessary conjugates.
- Use `repeat=False` for one simultaneous FeynRules-style definitions block; use `repeat=True` only for intentionally dependent stages.
- Run `find_source_basis_occurrences(...)` and `validate_compiled_index_multiplicities(...)` after a nontrivial transformation pipeline.

The production examples are in `models/SM/SM.py` and `models/SM/SM_support.py`; focused behavior and edge cases are covered by `tests/test_field_transformations.py` and `models/SM/tests/test_standard_model.py`.